# Visual reasoning · SigLIP2, geometry, K=16 slots

Валидация сравнивает геометрию, ансамбль со слотами, альтернативную обработку пространственной опоры и дообучение слотов. Финальный режим обучает выбранную модель с начала и экспортирует `final_submission.csv`.

Kaggle inputs содержат файлы соревнования, `visual-reasoning-source.zip` и опубликованные веса OpenCLIP `open_clip_model.safetensors` для `timm/ViT-B-16-SigLIP2-256`. Выберите **GPU T4 x2**, **Internet Off**.

In [ ]:
RUN_MODE = "validate"    # "validate" или "final"
VARIANT = "baseline"     # для final: "geometry", "baseline", "marginal", "finetune"
REFIT_TRAIN_VAL = False  # опция для финала после выбора модели на holdout

SEED = 42
GPUS = 2
IMAGE_BATCH = 128        # изображений на GPU при прогоне SigLIP2
TRAIN_BATCH = 256        # строк, то есть 128 пар, на GPU
SLOT_BATCH = 64          # изображений на GPU при реконструкции
SLOT_STEPS = 1600
EPOCHS = (18, 18, 18, 18, 6)  # пары, подсчёт, геометрия, слоты, дообучение
SCHEDULE_EPOCHS = None   # в final укажите исходный EPOCHS из validate для того же расписания

# Оставьте None для поиска входов или укажите конкретные пути.
COMPETITION_DATA = None
SOURCE = None
SIGLIP2_WEIGHTS = None
WHEELS = None           # папка с совместимыми .whl, если их нет в образе Kaggle

## Входы и зависимости

Код распаковывается в рабочий каталог. Веса SigLIP2 берутся из подключённого input. Слоты и остальные обучаемые части создаются в этом прогоне.

In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time
from zipfile import ZipFile

notebook_start = time.monotonic()
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

kaggle = Path("/kaggle/input").is_dir()
inputs = Path("/kaggle/input") if kaggle else Path.cwd()
output_dir = Path("/kaggle/working") if kaggle else Path.cwd()
scratch = Path("/kaggle/temp") if kaggle else output_dir / "artefacts"
run_id = f"kaggle_{time.time_ns()}"
run_dir = scratch / run_id

def unpack(path, destination):
    with ZipFile(path) as archive:
        for member in archive.infolist():
            if not (destination / member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError(f"Invalid archive path: {member.filename}")
        archive.extractall(destination)

def choose(label, explicit, candidates):
    if explicit is not None:
        path = Path(explicit).resolve()
        if not path.exists():
            raise FileNotFoundError(f"{label}: {path}")
        return path
    paths = sorted({p.resolve() for p in candidates})
    if len(paths) != 1:
        raise RuntimeError(f"{label}: укажите путь в настройках. Найдено: {paths}")
    return paths[0]

data_candidates = (p.parent for p in inputs.rglob("train.csv")
                   if (p.parent / "val.csv").is_file() and (p.parent / "train").is_dir())
data = choose("COMPETITION_DATA", COMPETITION_DATA, data_candidates)

bundled = list(inputs.rglob("visual-reasoning-kaggle-input.zip")) if kaggle else []
if bundled and not list(inputs.rglob("open_clip_model.safetensors")) and SIGLIP2_WEIGHTS is None:
    inputs = run_dir / "input"
    unpack(choose("Kaggle input ZIP", None, bundled), inputs)
if SOURCE is None and not kaggle and Path("scripts/kaggle_slots.py").is_file():
    source = Path.cwd()
else:
    expanded = [p.parent.parent for p in inputs.rglob("kaggle_slots.py")
                if (p.parent.parent / "configs/config.yaml").is_file()]
    source = choose("SOURCE", SOURCE, expanded or list(inputs.rglob("visual-reasoning-source.zip")))
weights = choose("SIGLIP2_WEIGHTS", SIGLIP2_WEIGHTS, inputs.rglob("open_clip_model.safetensors"))

project = run_dir / "source"
if source.is_dir():
    for name in ("src", "configs", "scripts"):
        shutil.copytree(source / name, project / name, ignore=shutil.ignore_patterns("__pycache__"))
else:
    project.mkdir(parents=True)
    unpack(source, project)
if not (project / "scripts/kaggle_slots.py").is_file():
    raise FileNotFoundError("scripts/kaggle_slots.py в исходниках")

check_code = (
    "import hydra, numpy, open_clip, polars, scipy, sklearn, timm, torch, torchvision; "
    "assert open_clip.get_model_config('ViT-B-16-SigLIP2-256') is not None, 'OpenCLIP без SigLIP2-256'"
)
check = subprocess.run([sys.executable, "-c", check_code], capture_output=True, text=True)
if check.returncode:
    wheel_dirs = [p.parent for p in inputs.rglob("open_clip_torch-*.whl")]
    wheel_zips = list(inputs.rglob("visual-reasoning-wheels.zip"))
    if not wheel_dirs and not wheel_zips and WHEELS is None:
        raise RuntimeError("Подключите offline wheels для недостающих зависимостей.\n" + check.stderr)
    wheels = choose("WHEELS", WHEELS, wheel_dirs or wheel_zips)
    if wheels.is_file():
        with ZipFile(wheels) as archive:
            if any(Path(name).name != name or not name.endswith(".whl") for name in archive.namelist()):
                raise ValueError("Ожидается ZIP с файлами .whl в корне")
            wheels = run_dir / "wheels"
            archive.extractall(wheels)
    packages = ["hydra-core==1.3.2", "open-clip-torch==4.2.2", "polars==2.1.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links",
                    str(wheels), *packages], check=True)
    subprocess.run([sys.executable, "-c", check_code], check=True)

print("Данные:", data)
print("Веса:", weights)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)

## Обучение и предсказания

Две карты параллельно извлекают признаки оригинальных и отражённых сцен. Предобучение слотов идёт одновременно с обучением выбора подписи и счётчика. Затем геометрическая и слотовая ветки обучаются на разных картах.

В `validate` train используется для всех этапов обучения, val для метрик и выбора эпох. В `final` число эпох фиксировано и файл test открывается после обучения. Обе карты предсказывают для непересекающихся групп сцен. Подписи-близнецы одной сцены остаются на одной карте.

In [ ]:
work = run_dir / run_id
submission_path = output_dir / "final_submission.csv"
command = [
    sys.executable, "-u", "-m", "scripts.kaggle_slots",
    "--data", str(data), "--weights", str(weights), "--work", str(work),
    "--output", str(submission_path), "--mode", RUN_MODE, "--variant", VARIANT,
    "--seed", str(SEED), "--gpus", str(GPUS), "--batch", str(TRAIN_BATCH),
    "--extract-batch", str(IMAGE_BATCH), "--slot-batch", str(SLOT_BATCH),
    "--slot-steps", str(SLOT_STEPS), "--epochs", *map(str, EPOCHS),
]
if REFIT_TRAIN_VAL:
    command.append("--refit")
if SCHEDULE_EPOCHS is not None:
    command.extend(["--schedule-epochs", *map(str, SCHEDULE_EPOCHS)])

report_dir = output_dir / "visual_reasoning_reports" / run_id
report_dir.mkdir(parents=True, exist_ok=True)
try:
    process = subprocess.Popen(command, cwd=project, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (report_dir / "run.log").open("w", encoding="utf-8") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if process.wait() != 0:
        raise RuntimeError(f"Прогон завершился с ошибкой. Логи: {report_dir}")
finally:
    for log_path in work.glob("*.log"):
        shutil.copyfile(log_path, report_dir / log_path.name)

report = json.loads((work / "report.json").read_text())
report["notebook_total_min"] = round((time.monotonic() - notebook_start) / 60, 2)
(report_dir / "report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

In [ ]:
import pandas as pd

if RUN_MODE == "validate":
    cols = ["auc", "accuracy", "pair_accuracy", "auc/counting", "auc/spatial_counting"]
    display(pd.DataFrame.from_dict(report["holdout"], orient="index")[cols].round(6))
    display(pd.DataFrame.from_dict(report["image_bootstrap_vs_baseline"], orient="index"))
    print("Эпохи для финала по этапам:", tuple(report["epochs_for_refit"]))
else:
    print("Файл:", submission_path)
    print("Строк:", report["submission"]["rows"])
    print("Время ≤ 50 мин:", report["notebook_total_min"] <= 50)

print("Полное время, мин:", report["notebook_total_min"])
print("Отчёт и логи:", report_dir)